# Deliverable 4 — Demo EN VIVO en Colab (≤ 3 min): Base vs `Redactor` QLoRA

Cuaderno diseñado para cumplir estrictamente la pauta del **video demostrativo**:
1. Muestra qué agente fue fine-tuneado (**solo `Redactor`** sobre `Qwen/Qwen3-4B` con `Pablox0333/qwen3-4b-redactor-udec-d4`) y verifica que los otros tres (`Enrutador`, `Recuperador`, `Verificador`) mantienen pesos base (`model.disable_adapter()`).
2. Ejecuta **EN VIVO sobre entradas reales** (`dificil_25:H07` en Base vs FT) mostrando los **mensajes intermedios reales** entre agentes.
3. Expone transparentemente un **caso de fallo real** (`dificil_25:H06` y `dificil_25:H03`) con su traza guardada.

**Recomendación de grabación (≤ 3 min):** Ejecuta la **Sección 1 (Setup + precarga del modelo en GPU T4/L4/A100)** *antes* de empezar a grabar pantalla; luego inicia OBS/grabador y corre la **Sección 2 en vivo** (si aceleras el tiempo de espera durante la inferencia en edición, coloca un rótulo explícito en pantalla como `[x6 durante inferencia]` sin cortar nunca las trazas).

## 1. (ANTES DE GRABAR) Instalar dependencias, clonar rama `deliverable-4` y pre-cargar modelo en GPU

In [ ]:
%pip install -q -U "transformers>=4.51" peft bitsandbytes accelerate "huggingface-hub>=0.30,<2.0" sentence-transformers

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Activa GPU (T4/L4) en Entorno de ejecución antes de grabar la demo en vivo."

REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
RAMA_GIT = "deliverable-4"
WORKDIR = Path("/content/d4_video_demo")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", RAMA_GIT, REPO_URL, str(WORKDIR)],
    check=True,
)

for sub in ("scripts", "video"):
    ruta = str(WORKDIR / "Deliverable 4" / sub)
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

import evaluar_d4 as ed
import mostrar_demo_d4 as demo

print("Commit activo:", subprocess.check_output(["git", "-C", str(WORKDIR), "rev-parse", "--short", "HEAD"], text=True).strip())
ed.verificar_prompts_d4()
CORPUS = ed.construir_corpus_d3(use_e5=True)
MODEL, TOKENIZER, DEVICE_INFO, _ = ed._cargar_modelo_y_adaptador(ed.MODELO_BASE_DEFAULT, ed.HF_ADAPTER_DEFAULT)
AISLAMIENTO = ed.verificar_aislamiento_adaptador(MODEL, TOKENIZER)
print("Modelo cargado en:", DEVICE_INFO, "| Aislamiento LoRA verificado max|Δlogits| =", AISLAMIENTO["max_abs_logit_diff"])

## 2. (GRABAR EN VIVO) Comparación Base vs Fine-tuned sobre `dificil_25:H07` con mensajes intermedios
Corre en vivo el pipeline de 4 agentes sobre la misma pregunta (`H07`), primero con `LoRA OFF` en los 4 roles y luego con `LoRA ON` solo en el `Redactor`, mostrando los mensajes intermedios reales.

In [ ]:
from IPython.display import HTML, display

OUT_LIVE = WORKDIR / "Deliverable 4" / "video" / "live_runs"
PREG_H07 = ed.cargar_preguntas_evaluacion(filtro_qids={"dificil_25:H07"})
GEN_BASE = ed.AdaptadorRedactorGenerator(MODEL, TOKENIZER, greedy=False, solo_base=True)
GEN_FT = ed.AdaptadorRedactorGenerator(MODEL, TOKENIZER, greedy=False, solo_base=False)

comp_live = ed.ejecutar_comparativa_base_vs_finetuned(
    PREG_H07,
    CORPUS,
    GEN_BASE,
    GEN_FT,
    OUT_LIVE / "comparativa_base_vs_finetuned.json",
)
from collections import Counter

conteo_ft = Counter((c["agente"], c["usa_adaptador"]) for c in GEN_FT.telemetria)
print("Aislamiento Base (esperado 0 ON):", sum(1 for c in GEN_BASE.telemetria if c["usa_adaptador"]))
print("Aislamiento FT llamadas (agente, usa_adaptador):", {f"{ag}:ON={on}": n for (ag, on), n in sorted(conteo_ft.items())})
display(HTML(demo.renderizar_comparativa_html(comp_live[0])))

## 2b. Alternativa CLI directa (`mostrar_demo_d4.py --live`)
Útil si prefieres grabar terminal en vivo sin celdas previas (tarda ~4-5 min incluyendo carga de pesos).

In [ ]:
import subprocess, sys

EJECUTAR_CLI_LIVE = False  # Cambia a True si quieres disparar la versión CLI desde cero
if EJECUTAR_CLI_LIVE:
    subprocess.run(
        [
            sys.executable,
            str(WORKDIR / "Deliverable 4" / "video" / "mostrar_demo_d4.py"),
            "--live",
            "--qid", "dificil_25:H07",
            "--output-dir", "/content/live_demo_h07",
        ],
        check=True,
    )

## 3. (GRABAR EN PANTALLA) Transparencia de fallo con traza guardada (`dificil_25:H06` y `dificil_25:H03`)

In [ ]:
fallos = demo.cargar_fallos(WORKDIR / "Deliverable 4" / "evaluacion" / "casos_fallo_y_regresion.json")
print(demo.formatear_fallo_texto(fallos["dificil_25:H06"], mostrar_traza=True))